# Lecture 6 Examples and Case — Data Cleaning and Text Preparation

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 6  
**Goal:** Clean transparently, flag problems, prepare text, and record every transformation.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/examples/L06_examples_data_cleaning_text_preparation.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Assess and clean a dataset transparently, preserve the raw data, record transformations, and prepare a small text field.


## Setup

The notebook uses pandas and NLTK's regular-expression tokenizer. It does not download language models or hidden resources.

### Running this notebook on a local computer

1. Download or clone the course repository.
2. Open a terminal in the repository folder.
3. Create and activate a virtual environment.
4. Install the shared requirements with `python -m pip install -r requirements.txt`.
5. Start Jupyter with `python -m jupyter lab`.

Packages used directly in this notebook: `nltk pandas`

## Steps

### 1. Load and preserve the raw data


In [1]:
from pathlib import Path  # Imports Path so the notebook can find a local course file when available.
import pandas as pd  # Imports pandas for reading and working with table-shaped data.
remote_data_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/monthly_service_report.csv"  # Stores the public GitHub address used by Google Colab.
local_data_candidates = [Path("data/monthly_service_report.csv"), Path("../../data/monthly_service_report.csv")]  # Lists possible local paths used during validation.
data_source = next((path for path in local_data_candidates if path.exists()), remote_data_url)  # Chooses a local file when present and otherwise uses GitHub.
raw_data = pd.read_csv(data_source)  # Reads the CSV file into a pandas DataFrame.
print(raw_data.head())  # Prints a small preview so we can confirm that loading worked.


   record_id report_month          city service_type  cases_received  \
0       1001   2026-01-01    Copenhagen      Housing             120   
1       1002   2026-01-01   copenhagen     Transport              85   
2       1003   2026-01-01       AALBORG      Housing              -3   
3       1004   2026-02-01    Koebenhavn   Employment              74   
4       1004   2026-02-01    Koebenhavn   Employment              74   

   cases_resolved resolution_days  satisfaction_score  \
0             112             5.1                 4.2   
1              90             3.2                 4.6   
2               0             8.4                 3.1   
3              68             4.0                 4.0   
4              68             4.0                 4.0   

                                  feedback  
0           Helpful staff and clear answer  
1  Quick answer but the form was confusing  
2                  Long wait for an answer  
3                   The guidance was clear  

In [2]:
clean_data = raw_data.copy()  # Creates a separate working copy so the original observations remain unchanged.
cleaning_log = []  # Creates an empty list that will document every transformation.


### 2. Standardise text categories


In [3]:
clean_data["city"] = clean_data["city"].astype("string").str.strip().str.lower()  # Removes surrounding spaces and normalises city capitalisation.
clean_data["city"] = clean_data["city"].replace({"koebenhavn": "copenhagen"})  # Maps one known spelling variant to the agreed category.
clean_data["city"] = clean_data["city"].str.title()  # Applies readable title-style capitalisation after mapping.
clean_data["service_type"] = clean_data["service_type"].astype("string").str.strip().str.title()  # Standardises the service category labels.
cleaning_log.append("Standardised city and service labels; mapped Koebenhavn to Copenhagen.")  # Records the reason and action in the log.
print(sorted(clean_data["city"].dropna().unique().tolist()))  # Displays the cleaned city categories for review.


['Aalborg', 'Copenhagen']


### 3. Convert dates and numeric columns


In [4]:
clean_data["report_month"] = pd.to_datetime(clean_data["report_month"], errors="coerce")  # Converts valid dates and marks impossible dates as missing.
numeric_columns = ["cases_received", "cases_resolved", "resolution_days", "satisfaction_score"]  # Lists columns that should contain numbers.
for numeric_column in numeric_columns:  # Visits each expected numeric column.
    clean_data[numeric_column] = pd.to_numeric(clean_data[numeric_column], errors="coerce")  # Converts valid numbers and marks invalid text as missing.
cleaning_log.append("Converted dates and numeric columns; invalid values became missing for review.")  # Documents the conversion policy.
print(clean_data[numeric_columns].dtypes)  # Displays the resulting numeric data types.


cases_received          int64
cases_resolved          int64
resolution_days       float64
satisfaction_score    float64
dtype: object


### 4. Remove confirmed duplicates


In [5]:
rows_before_deduplication = len(clean_data)  # Stores the row count before removing exact duplicates.
clean_data = clean_data.drop_duplicates().copy()  # Removes only rows that are exact duplicates across every column.
rows_after_deduplication = len(clean_data)  # Stores the row count after the operation.
removed_duplicate_count = rows_before_deduplication - rows_after_deduplication  # Calculates how many rows were removed.
cleaning_log.append(f"Removed {removed_duplicate_count} exact duplicate row(s).")  # Records the exact effect of the operation.
print(removed_duplicate_count)  # Displays the number of confirmed duplicates removed.


1


### 5. Flag implausible values instead of silently deleting them


In [6]:
clean_data["case_count_problem"] = (clean_data["cases_received"] < 0) | (clean_data["cases_resolved"] < 0) | (clean_data["cases_resolved"] > clean_data["cases_received"])  # Flags simplified case-count rule failures.
clean_data["score_problem"] = ~clean_data["satisfaction_score"].between(1, 5) & clean_data["satisfaction_score"].notna()  # Flags non-missing satisfaction scores outside the stated scale.
clean_data["date_problem"] = clean_data["report_month"].isna()  # Flags dates that could not be parsed.
problem_rows = clean_data[clean_data[["case_count_problem", "score_problem", "date_problem"]].any(axis=1)]  # Selects rows with at least one definite rule failure.
print(problem_rows[["record_id", "case_count_problem", "score_problem", "date_problem"]])  # Displays a focused audit table rather than deleting observations.


   record_id  case_count_problem  score_problem  date_problem
1       1002                True          False         False
2       1003                True          False         False
8       1008               False           True         False
9       1009               False          False          True


### 6. Prepare text with NLTK


In [7]:
from nltk.tokenize import RegexpTokenizer  # Imports a tokenizer that works without downloading additional NLTK files.
tokenizer = RegexpTokenizer(r"[A-Za-z]+")  # Defines tokens as one or more alphabetic characters.
stop_words = {"a", "and", "the", "to", "was"}  # Defines a short transparent stop-word list for this teaching example.
feedback_text = clean_data["feedback"].dropna().astype(str).str.cat(sep=" ")  # Combines non-missing feedback into one text string.
raw_tokens = tokenizer.tokenize(feedback_text.lower())  # Lowercases the text and splits it into alphabetic tokens.
prepared_tokens = [token for token in raw_tokens if token not in stop_words]  # Removes only the explicitly listed stop words.
print(prepared_tokens[:20])  # Displays a bounded sample of the prepared tokens.


['helpful', 'staff', 'clear', 'answer', 'quick', 'answer', 'but', 'form', 'confusing', 'long', 'wait', 'for', 'an', 'answer', 'guidance', 'clear', 'several', 'steps', 'were', 'difficult']


### 7. Review the transformation log


In [8]:
for log_number, log_entry in enumerate(cleaning_log, start=1):  # Numbers each recorded transformation from one.
    print(f"{log_number}. {log_entry}")  # Displays the transformation log in order.
print(f"Raw rows: {len(raw_data)}; cleaned rows: {len(clean_data)}")  # Compares row counts before and after cleaning.
print(clean_data.isna().sum())  # Displays remaining missingness that still requires an analytical decision.


1. Standardised city and service labels; mapped Koebenhavn to Copenhagen.
2. Converted dates and numeric columns; invalid values became missing for review.
3. Removed 1 exact duplicate row(s).
Raw rows: 16; cleaned rows: 15
record_id             0
report_month          1
city                  0
service_type          0
cases_received        0
cases_resolved        0
resolution_days       1
satisfaction_score    1
feedback              1
case_count_problem    0
score_problem         0
date_problem          0
dtype: int64


## Checks

The raw DataFrame remains unchanged. One exact duplicate is removed. Invalid dates and non-numeric text become missing, while rule failures are flagged and retained for review.


## Next Steps

Use the exercise notebook to justify each cleaning choice. A cleaning command is not a justification by itself.
